# Mechanism of the c law (1/c = 1.36 + 0.70 lead) and what it implies for kappa

## Question and why it matters
In chaotic generalised Langton's ants, c = births per boundary step is almost a function of the rule's opening same-letter run (`lead`): 1/c = 1.363 + 0.696 lead on the census. This notebook asks why, whether it is a fresh-ground (edge) effect, and whether the earlier "lead raises kappa at fixed area" finding is real.
Registrations, failed predictions and deviations: `README.md`; thinking: `NOTEBOOK.md`; summary in `notes/open-questions-hypotheses.md`. Predecessor: `experiments/archive/2026-10-kappa-chiral-edge`.

## Assumptions and definitions
- Boundary step: starts on a visited cell with an unvisited 4-neighbour (as `src/probe.rs`, inner-hole rims included). Birth: step onto a never-visited cell. c = births / boundary steps. Window = last octave of 2^24 steps on a 4096 board unless stated.
- `src/edge_trace.rs` is the instrument; it agrees exactly with `probe_walker` and a HashMap brute force (`tests/edge_trace.rs`). **Held** (calibration: rotor c 0.4977 vs 0.498, random walk 0.448 vs 0.44).
- Assumption: Stage 2/3 filter "kept" = not escaped, births >= 200, 1.3 <= area/area_t0 <= 1.9 (blob-like). **Untested** beyond the counts printed below.

## Provenance

In [1]:
import hashlib, json, subprocess, datetime
from pathlib import Path
import numpy as np, polars as pl
SEED = 20261019
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
R = ROOT / "experiments/archive/2026-10-c-law-mechanism/results"
names = ["stage1_profile.json", "stage1_k3.json", "stage2_analysis.json", "stage2c_analysis.json", "stage3_analysis.json", "stage4_analysis.json",
         "stage2_decompose.txt", "p1/p1_pairs.csv", "PROVENANCE.md"]
assert all((R / n).exists() for n in names), [n for n in names if not (R / n).exists()]
sha = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, capture_output=True, text=True).stdout.strip() or "unresolvable from WSL; see PROVENANCE.md: base f0b96b7, code b7b4758, results 8cc447f"
print("git", sha, "| seed", SEED, "| run", datetime.date.today().isoformat())
for n in names:
    print(hashlib.sha256((R / n).read_bytes()).hexdigest()[:16], n)
J = lambda n: json.loads((R / n).read_text())

git unresolvable from WSL; see PROVENANCE.md: base f0b96b7, code b7b4758, results 8cc447f | seed 20261019 | run 2026-10-19
5116b3d2c5953e47 stage1_profile.json
0b528f84dfcf315d stage1_k3.json
c3737dfe00c82ebb stage2_analysis.json
477b2bee88362cb1 stage2c_analysis.json
94efd397637b23ba stage3_analysis.json
09cff85b6d4be3d9 stage4_analysis.json
7afaed06229ed73e stage2_decompose.txt
0373afcab7343812 p1/p1_pairs.csv
13f078a930c668a8 PROVENANCE.md


## Data and method
Stage 1: per-colour profile of boundary steps from `edgestats_sample.csv` (207 stationary blob rules). Stage 2: tracer on 207 rules, 600 new validation rules, calibration walkers, and a **flat torus band** (256 x 1024, fresh rows 0..511, i.i.d. interior) with an interior ablation. Stage 3: `SparseBgAnt` dose-response, 60 rules x densities. Stage 4: kappa (collider check, growth identity, lead-preserving early flips run with the unchanged `probe 27 8192` pipeline). Heavy computation is in Rust (`examples/c_law_mechanism_trace.rs`, `probe`).

## Results
### 1. c is a local fresh-ground property

In [2]:
s2, s2c = J("stage2_analysis.json"), J("stage2c_analysis.json")
print("flat band (i.i.d. interior): 1/c = %.3f + %.3f lead; corr^2 with census c %.4f; median c_band/c_census %.4f" % (s2["S"]["band_intercept"], s2["S"]["band_slope"], s2["S"]["corr2"], s2["S"]["ratio_median"]))
for k, nm in zip("ZYH", ["all-zero", "young colours", "old colours"]):
    v = s2c[k]; print(f"interior {nm:14s}: median |c/c_U - 1| = {v['median_rel']:.4f}, 1/c = {v['intercept']:.3f} + {v['slope']:.3f} lead, kept {v['n']}/60")

flat band (i.i.d. interior): 1/c = 1.399 + 0.678 lead; corr^2 with census c 0.9986; median c_band/c_census 0.9983
interior all-zero      : median |c/c_U - 1| = 0.0029, 1/c = 1.409 + 0.676 lead, kept 59/60
interior young colours : median |c/c_U - 1| = 0.0031, 1/c = 1.412 + 0.674 lead, kept 58/60
interior old colours   : median |c/c_U - 1| = 0.0028, 1/c = 1.382 + 0.682 lead, kept 58/60


A flat front in a random interior reproduces the census c rule by rule to 0.2%, and even an all-colour-0 interior changes it by 0.3%. It does **not** explain the numbers 1.4 and 0.68; it shows they are properties of the rule on fresh ground (not of curvature, holes or the ant-built bulk).

### 2. Where the boundary steps go

In [3]:
p = J("stage1_profile.json")
for m, v in p["rk_by_lead"].items():
    print(f"lead {m} (n={v['n']:3d}): r_k = " + " ".join(f"{x:.2f}" for x in v["mean"]))
print(); print((R / "stage2_decompose.txt").read_text())
print("lag-4 share of circulating reads: %.3f; killing reads: %.3f; chains q1,q2,q3 = %s; Little median E[N_b]/(1/c) = %.3f" % (s2["L1"]["frac_lag4"], s2["L2"]["frac_lag4"], np.round(s2["C1"]["q"][:3], 3), s2["N1"]["median_ratio"]))

lead 1 (n=120): r_k = 0.97 0.72 0.26 0.09 0.04
lead 2 (n= 39): r_k = 0.98 0.85 0.58 0.14 0.07 0.03
lead 3 (n= 29): r_k = 0.99 0.89 0.74 0.53 0.09 0.04 0.02
lead 4 (n= 12): r_k = 0.98 0.89 0.79 0.68 0.53 0.08 0.04 0.02
lead 5 (n=  3): r_k = 0.99 0.98 0.87 0.73 0.61 0.49 0.06 0.04 0.01
lead 6 (n=  3): r_k = 1.00 0.93 0.86 0.78 0.72 0.65 0.56 0.09 0.04

207 sample rules (stationary blob), lead capped at 6
  first reads    per birth = 1.434 + 0.227 lead   (share of the lead slope: 0.32)
  lag-4 returns  per birth = -0.202 + 0.394 lead   (share of the lead slope: 0.55)
  other returns  per birth = 0.121 + 0.092 lead   (share of the lead slope: 0.13)
  total = 1/c    per birth = 1.352 + 0.712 lead   (share of the lead slope: 1.00)
  lead 1 (n=120): first 1.608  lag4 0.300  other 0.211  total 2.120
  lead 2 (n=39): first 1.983  lag4 0.410  other 0.293  total 2.685
  lead 3 (n=29): first 2.352  lag4 0.566  other 0.408  total 3.325
  lead 4 (n=12): first 2.149  lag4 1.324  other 0.568  total 4.

1/c is exactly the number of boundary reads per born cell (Little's law holds to 1.6%); 78% of circulating reads are 4-step returns (2x2 loops); runs of consecutive births never exceed 3. The registered "flat plateau" and "geometric chain" pictures failed. It does **not** derive a or b: the split is descriptive, and the "first reads" bucket in the decomposition also contains returns with lag > 40.

### 3. Sparse background: the slope is a fresh-ground effect

In [4]:
s3 = J("stage3_analysis.json")
print(pl.DataFrame([(float(d), v["a"], v["b"], v["mix_b"]) for d, v in s3["fit"].items() if not d.startswith("secondary")], schema=["density", "a", "b_obs", "b_mixture"], orient="row"))
print("B3' mixture R2 %.3f median rel err %.3f (null no density effect: %.3f / %.3f)" % (s3["B3p"]["r2"], s3["B3p"]["median_rel"], s3["B3p"]["null_d0_r2"], s3["B3p"]["null_d0_median_rel"]))

shape: (5, 4)
┌─────────┬──────────┬──────────┬───────────┐
│ density ┆ a        ┆ b_obs    ┆ b_mixture │
│ ---     ┆ ---      ┆ ---      ┆ ---       │
│ f64     ┆ f64      ┆ f64      ┆ f64       │
╞═════════╪══════════╪══════════╪═══════════╡
│ 0.0     ┆ 1.396906 ┆ 0.676665 ┆ 0.67778   │
│ 0.02    ┆ 1.439341 ┆ 0.645842 ┆ 0.664844  │
│ 0.05    ┆ 1.488665 ┆ 0.604713 ┆ 0.64544   │
│ 0.1     ┆ 1.537412 ┆ 0.548943 ┆ 0.6131    │
│ 0.2     ┆ 1.618568 ┆ 0.45025  ┆ 0.548419  │
└─────────┴──────────┴──────────┴───────────┘
B3' mixture R2 0.869 median rel err 0.067 (null no density effect: 0.613 / 0.102)


The lead slope falls monotonically with background density and a parameter-free single-cell mixture tracks it (observed falls slightly faster). The intercept rises, not falls. It does **not** reach d = 1: above d ~ 0.7 the ant leaves the board, so there is no blob regime there and the registered B1 was untestable.

### 4. Forward models of c

In [5]:
k3 = J("stage1_k3.json"); v = s2["V"]
print("census held-out R2(ln c): lead-only %.3f, K3 %.3f, K4 %.3f" % (k3["split"]["lead_only_r2_lnc"], k3["split"]["K3_r2_lnc_B"], s2["K4_census"]["r2_lnc_B"]))
print("fresh 176 R2(c): lead law %.3f, K3 %.3f" % (k3["fresh"]["lead_law_r2_c"], k3["fresh"]["K3_r2_c"]))
print("new validation rules (n=%d) R2(c): lead %.3f, K3 %.3f, K4 %.3f" % (v["n"], v["lead"]["r2"], v["K3"]["r2"], v["K4"]["r2"]))

census held-out R2(ln c): lead-only 0.842, K3 0.875, K4 0.863
fresh 176 R2(c): lead law 0.877, K3 0.913
new validation rules (n=378) R2(c): lead 0.711, K3 0.785, K4 0.800


A 3-parameter survival kernel over the first 12 letters beats the lead law out of sample by 0.04-0.07 in R^2, not by enough to reach the 0.95 of an 8-letter lookup. It does **not** explain why letters 5-8 matter; none of my few-parameter kernels did.

### 5. Kappa

In [6]:
s4 = J("stage4_analysis.json"); a = s4["4a"]
print("176 fresh rules: conditional lead coef %.3f vs no-direct-effect null %.3f %s; with ln c: %.3f %s" % (a["obs_lead"], a["null_lead_mean"], np.round(a["null_lead_ci"], 3), a["with_c"]["lead"], np.round(a["with_c"]["ci"], 3)))
print("growth fit ln A22 = %.2f + %.3f ln c + %.3f ln kappa" % tuple(a["growth_fit"]))
b = s4["4b"]; print("Part B pairs: g=%.3f" % b["g"], {k: (round(x["mean_dA_obs"], 3), round(x["mean_dA_pred"], 3)) for k, x in b["arms"].items()})
c = s4["4c"]; print("lead-preserving flips: %d pairs, mean d %.3f %s, area-adjusted residual %.3f" % (c["n_pairs"], c["mean_d"], np.round(c["ci"], 3), c["resid_mean"]))
pp = pl.read_csv(R / "p1/p1_pairs.csv").with_columns(flip=pl.struct("base", "idx").map_elements(lambda s: s["base"][s["idx"]] + ">" + ("R" if s["base"][s["idx"]] == "L" else "L"), return_dtype=pl.String))
print(pp.group_by("flip").agg(n=pl.len(), mean_d=pl.col("d").mean(), sd_d=pl.col("d").std()))

176 fresh rules: conditional lead coef 0.148 vs no-direct-effect null 0.145 [0.137 0.153]; with ln c: 0.014 [-0.045  0.062]
growth fit ln A22 = 11.45 + 0.687 ln c + 0.807 ln kappa
Part B pairs: g=0.805 {'E': (-0.292, -0.318), 'S': (0.013, 0.05), 'C': (-0.0, -0.004)}
lead-preserving flips: 177 pairs, mean d -0.007 [-0.078  0.06 ], area-adjusted residual 0.007
shape: (2, 4)
┌──────┬─────┬──────────┬──────────┐
│ flip ┆ n   ┆ mean_d   ┆ sd_d     │
│ ---  ┆ --- ┆ ---      ┆ ---      │
│ str  ┆ u32 ┆ f64      ┆ f64      │
╞══════╪═════╪══════════╪══════════╡
│ R>L  ┆ 79  ┆ -0.20685 ┆ 0.359488 │
│ L>R  ┆ 98  ┆ 0.153409 ┆ 0.419629 │
└──────┴─────┴──────────┴──────────┘


Conditioning on ln A22 (a function of c and kappa, residual sd 0.09) manufactures the +0.15 "lead effect on kappa"; a null with no direct effect reproduces it and adding ln c removes it. The E/S area asymmetry of the earlier mutants follows from the c law. Lead-preserving flips change kappa by zero on average. It does **not** show that lead has no causal effect on kappa: single flips move kappa by about +/-0.2 with sign set by letter direction (post hoc split), and the S and E arms are confounded with that direction.

## Caveats and controls
- Registered failures: D1-D4 (flat plateau), C1 (geometric chains), K3-1, K4 (all), V1, S5, L2, B5; B1 untestable. Deviations: random-walk calibration at 2^20, band window rule, Stage 3 restated cohort, P-4 NaN fix. All in README/NOTEBOOK.
- Timestamps in early NOTEBOOK entries were estimated, not read from the clock; real times and commits are in `results/PROVENANCE.md`. The edgestats sample used K = 23 (recovered).
- Survivor selection limits Stage 3 (d = 0.4) and Stage 4c (retention 0.65).

## What this changes / next steps
The c law is a property of the rule on fresh ground (flat band, interior ablation, sparse background). Open: derive a = 1.4, b = 0.68 (loop-formation probability x 4 steps / births per chain is the lead candidate); why letters 5-8 enter; the S/E direction confound for kappa (needs flips balanced for direction at fixed lead change); the blob regime boundary between d = 0.2 and 0.7.